<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo5/RAG_BM25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rank_bm25

In [ ]:
import getpass
import re
import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi

api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
client = OpenAI(api_key=api_key)

# -------------------------
# 1) CORPUS
# -------------------------
documentos = [
    "Política de teletrabajo: GlobalCorp permite 2 días por semana.",                         # Doc 0
    "WorkLife: guía para solicitar teletrabajo/home office desde el portal de RR.HH.",        # Doc 1
    "WL-742: URL oficial del portal de solicitudes -> wl-742.globalcorp.com (SSO requerido).",# Doc 2
    "WL-741: portal antiguo de home office/teletrabajo (NO usar) -> wl-741.globalcorp.com",   # Doc 3 (trampa: muy semántico, ID casi igual que el doc2)
]

# -------------------------
# 2) Tokenización BM25
# -------------------------
STOPWORDS_ES = {
    "el","la","los","las","un","una","unos","unas","de","del","al","a","en","y","o","u",
    "para","por","con","sin","que","como","donde","cuando","cual","cuál","es","son","se",
    "mi","mis","tu","tus","su","sus","lo","ya","no","si","sí","me","te","le","les","esta",
    "este","estos","estas","desde","entra","entrar","hacer"
}

def tokenize(text: str):
    # Mantiene letras con acentos + números + guiones
    tokens = re.findall(r"[a-záéíóúüñ0-9]+(?:-[a-z0-9]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS_ES]

tokenized_corpus = [tokenize(doc) for doc in documentos]
bm25 = BM25Okapi(tokenized_corpus)

# -------------------------
# 3) Embeddings
# -------------------------
def get_embeddings(texts, model="text-embedding-3-small"):
    resp = client.embeddings.create(input=texts, model=model)
    return np.array([x.embedding for x in resp.data], dtype=np.float32)

doc_emb = get_embeddings(documentos)
doc_emb = doc_emb / np.linalg.norm(doc_emb, axis=1, keepdims=True)

# -------------------------
# 4) Runner + salida ordenada (Top-k)
# -------------------------
def run_query(query: str, k: int = 4):
    q_tokens = tokenize(query)

    # BM25
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_rank = np.argsort(bm25_scores)[::-1][:k]

    # Coseno
    q_emb = get_embeddings([query])[0]
    q_emb = q_emb / np.linalg.norm(q_emb)
    cos_scores = doc_emb @ q_emb
    cos_rank = np.argsort(cos_scores)[::-1][:k]

    print("\n" + "="*90)
    print(f"CONSULTA: {query}")
    print(f"Tokens BM25: {q_tokens}")
    print("-"*90)

    def pretty(i):
        txt = documentos[i]
        return (txt[:80] + "…") if len(txt) > 81 else txt

    # Top BM25
    print("\nTOP BM25 (léxico / coincidencia exacta):")
    for pos, i in enumerate(bm25_rank, start=1):
        overlap = sorted(set(q_tokens).intersection(set(tokenized_corpus[i])))
        print(f"{pos:>2}. Doc {i} | score={bm25_scores[i]:.3f} | overlap={overlap}\n    {pretty(i)}")

    # Top Coseno
    print("\nTOP COSENO (semántico / significado):")
    for pos, i in enumerate(cos_rank, start=1):
        print(f"{pos:>2}. Doc {i} | score={cos_scores[i]:.3f}\n    {pretty(i)}")

# -------------------------
# 5) Consultas
# -------------------------

# Escenario donde BM25 suele GANAR: hay un ID exacto (WL-742)
run_query("¿Cómo entro a WL-742 para solicitar home office?")

# Escenario donde coseno suele GANAR: semántica/sinónimo (home office vs teletrabajo) sin ID
run_query("¿Cuántos días de home office permite la política?")

Introduce tu OpenAI API Key: ··········

CONSULTA: ¿Cómo entro a WL-742 para solicitar home office?
Tokens BM25: ['cómo', 'entro', 'wl-742', 'solicitar', 'home', 'office']
------------------------------------------------------------------------------------------

TOP BM25 (léxico / coincidencia exacta):
 1. Doc 2 | score=1.169 | overlap=['wl-742']
    WL-742: URL oficial del portal de solicitudes -> wl-742.globalcorp.com (SSO requ…
 2. Doc 1 | score=0.847 | overlap=['home', 'office', 'solicitar']
    WorkLife: guía para solicitar teletrabajo/home office desde el portal de RR.HH.
 3. Doc 3 | score=0.000 | overlap=['home', 'office']
    WL-741: portal antiguo de home office/teletrabajo (NO usar) -> wl-741.globalcorp…
 4. Doc 0 | score=0.000 | overlap=[]
    Política de teletrabajo: GlobalCorp permite 2 días por semana.

TOP COSENO (semántico / significado):
 1. Doc 3 | score=0.673
    WL-741: portal antiguo de home office/teletrabajo (NO usar) -> wl-741.globalcorp…
 2. Doc 2 | score=0.64